In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120
from pathlib import Path
import matplotlib.patches as patches

In [ ]:
f = uproot.open(Path("hits_offset.root"))
tree_key = next((k for k, v in f.classnames().items() if v.endswith("TTree")), None)
print("Tree:", tree_key)
if tree_key:
    tree = f[tree_key]
    print("\n".join(tree.keys()))

In [ ]:
# Set the ROOT file path here
root_file = Path('hits_offset.root')

if not root_file.exists():
    roots = sorted(Path('.').glob('*.root'))
    raise FileNotFoundError(f'{root_file} not found. Available: {[p.name for p in roots]}')

with uproot.open(root_file) as f:
    classnames = f.classnames()
    tree_key = next((k for k, v in classnames.items() if v.endswith('TTree')), None)
    if tree_key is None:
        raise RuntimeError(f'No TTree found in {root_file}. Keys: {list(classnames.keys())}')
    tree = f[tree_key]
    arrays = tree.arrays(library='np')

def _get(name):
    return arrays[name] if name in arrays else None

def _missing(n, value=np.nan):
    return np.full(n, value)

n = len(arrays[next(iter(arrays.keys()))]) if len(arrays) > 0 else 0

def _get_vec(base):
    arr = _get(base)
    if arr is not None:
        return arr
    x = _get(f"{base}_X")
    y = _get(f"{base}_Y")
    z = _get(f"{base}_Z")
    if x is None or y is None or z is None:
        return None
    return np.stack([x, y, z], axis=1)

pos = _get_vec('Position')
if pos is not None:
    print ("got Postion!")
if pos is None:
    print ("Postion empty, getting PrePosition")
    pos = _get_vec('PrePosition')
if pos is None:
    print ("PrePostion empty, getting PostPosition")
    pos = _get_vec('PostPosition')
if pos is None:
    print ("PostPostion empty, getting PrePositionLocal")
    pos = _get_vec('PrePositionLocal')
if pos is None:
    print ("All positions empty!")
    x_mm = y_mm = z_mm = _missing(n)
else:
    x_mm, y_mm, z_mm = pos[:, 0], pos[:, 1], pos[:, 2]

edep = _get('TotalEnergyDeposit')
if edep is None:
    edep = _get('Edep')
edep_keV = edep * 1000.0 if edep is not None else _missing(n)  # assuming MeV -> keV
process = _get('ProcessDefinedStep')
if process is None:
    process = _get('TrackCreatorProcess')

# Diagnostics for process availability
if process is None:
    print('WARNING: No ProcessName or TrackCreatorProcess in ROOT file')
else:
    process_series = pd.Series(process)
    process_series = pd.Series(process)
    empty_frac = (process_series.astype(str).str.len() == 0).mean()
    print(f'Empty process fraction: {empty_frac:.1%}')

volume = _get('TrackVolumeName')
if volume is None:
    volume = _get('VolumeName')

df = pd.DataFrame(
    {
        'event': _get('EventID') if _get('EventID') is not None else _missing(n),
        'run': _get('RunID') if _get('RunID') is not None else _missing(n),
        'thread': _get('ThreadID') if _get('ThreadID') is not None else _missing(n),
        'track': _get('TrackID') if _get('TrackID') is not None else _missing(n),
        'parent': _get('ParentID') if _get('ParentID') is not None else _missing(n),
        'particle': _get('ParticleName').astype(str) if _get('ParticleName') is not None else _missing(n, ''),
        'process': process.astype(str) if process is not None else _missing(n, ''),
        'edep_keV': edep_keV,
        'x_mm': x_mm,
        'y_mm': y_mm,
        'z_mm': z_mm,
        't_ns': _get('GlobalTime') if _get('GlobalTime') is not None else _missing(n),
        'volume': volume.astype(str) if volume is not None else _missing(n, ''),
        'copyno': _get('CopyNo') if _get('CopyNo') is not None else _missing(n),
    }
)

df['interaction_type'] = np.where(
    df['process'].str.contains('phot', case=False, na=False),
    'PE',
    np.where(df['process'].str.contains('compt', case=False, na=False), 'Compton', 'Other'),
)

In [ ]:
# Scatter plot of hit XY with pixel grid overlay

if 'df' not in globals():
    print('df not found; run the data-loading cell first.')
else:
    x = df['x_mm'].to_numpy()
    y = df['y_mm'].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # Subsample for speed if needed
    max_points = 200000
    if len(x) > max_points:
        rng = np.random.default_rng(0)
        idx = rng.choice(len(x), size=max_points, replace=False)
        x = x[idx]
        y = y[idx]

    pix = 3.0
    foil = 0.2
    n = 8
    pitch = pix + foil
    offset = (n - 1) * pitch / 2.0

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(x, y, s=2, alpha=0.3, color='black')

    # Draw pixel rectangles
    for ix in range(n):
        for iy in range(n):
            cx = ix * pitch - offset
            cy = iy * pitch - offset
            rect = patches.Rectangle(
                (cx - pix / 2.0, cy - pix / 2.0),
                pix,
                pix,
                fill=False,
                edgecolor='tab:blue',
                linewidth=0.5,
            )
            ax.add_patch(rect)

    # Draw ESR block outline (including outer foil)
    block_xy = n * pix + (n - 1) * foil + 2 * foil
    block = patches.Rectangle(
        (-block_xy / 2.0, -block_xy / 2.0),
        block_xy,
        block_xy,
        fill=False,
        edgecolor='tab:red',
        linewidth=1.0,
    )
    ax.add_patch(block)

    ax.set_aspect('equal', 'box')
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')
    ax.set_title('Hit XY with Pixel Grid Overlay')
    plt.tight_layout()
    plt.show()
